# AE3 - Transformación y enriquecimiento del dataset Olist

**Proyecto integrador - Etapa 2: transformación y enriquecimiento de datos**

Este notebook continúa el proyecto de la AE2 con el mismo dataset y el mismo DataFrame de ítems. Su flujo es:

**dataset de la AE2 → transformación y validación → enriquecimiento → respuestas a las preguntas de negocio → tablas y archivos exportados**

Al final exporta automáticamente, con rutas relativas:

| Archivo | Contenido |
|---|---|
| `AE3/datos/olist_items_preparado.csv` | Dataset transformado y enriquecido |
| `AE3/resultados/tabla_p1_categorias.csv` ... `tabla_p5_mensual.csv` | Una tabla de resultados por pregunta |
| `AE3/resultados/resumen_dataframe_final.txt` | Información descriptiva del DataFrame final |

| Ficha | Valor |
|---|---|
| Dataset | `bulutttt/olist-raw-data` (Hugging Face), copia del *Brazilian E-Commerce Public Dataset by Olist* |
| Revisión utilizada | `5acf1f37d7197d4979e714721a1ec13a7c3da728` (la misma de la AE2) |
| Split | `train` (único disponible) |
| Licencia | CC BY-NC-SA 4.0: uso no comercial con atribución. Los datos son públicos y se pueden compartir con atribución |
| Unidad de análisis | Ítem de orden (cada fila es un producto vendido dentro de una orden) |

## 1. Necesidad, preguntas de negocio y variables

**Necesidad (definida en la AE2):** el equipo comercial y de operaciones de un marketplace necesita conocer qué
categorías de producto y qué estados de Brasil concentran las unidades vendidas y los ingresos por venta de productos,
y cómo se comporta el tiempo de entrega en esos grupos, para priorizar decisiones comerciales y logísticas.

**Pregunta de análisis de la AE2:** ¿cómo se distribuyen las unidades vendidas y los ingresos por venta de productos entre
las categorías de producto y los estados del cliente en las órdenes de Olist registradas entre septiembre de 2016 y
septiembre de 2018, y cómo varía el tiempo de entrega entre esos grupos?

Para responderla, la pregunta se desagrega en cinco preguntas de negocio, una por cada objetivo específico 2 a 5 de la
AE2 (el objetivo 1, caracterizar el dataset, se cumplió en la AE2). El tiempo de entrega se separa en dos preguntas
(por estado y por categoría) para tener una tabla por pregunta.

| Pregunta de negocio | Variables necesarias | Resultado esperado |
|---|---|---|
| **P1.** ¿Qué categorías de producto concentran las unidades vendidas y los ingresos? | `categoria_producto`, `price`, `order_id` | Tabla por categoría: unidades, órdenes, ingresos (BRL), participación %, ingreso promedio por ítem y ranking |
| **P2.** ¿Cómo se distribuyen las unidades, los ingresos y el ingreso promedio por ítem entre los estados del cliente? | `customer_state`, `region_cliente`, `price`, `order_id` | Tabla por estado: unidades, órdenes, ingresos, participación % e ingreso promedio por ítem |
| **P3.** ¿Cómo varía el tiempo de entrega y el porcentaje de entregas tardías entre los estados del cliente? | `customer_state`, `region_cliente`, `dias_entrega`, `entrega_tardia` | Tabla por estado: ítems entregados, mediana y promedio de días, % de entregas tardías |
| **P4.** ¿Cómo varía el tiempo de entrega y el porcentaje de entregas tardías entre las categorías de producto? | `categoria_producto`, `dias_entrega`, `entrega_tardia` | Tabla por categoría con las mismas medidas de P3 |
| **P5.** ¿Cómo evolucionan mes a mes las unidades vendidas y los ingresos? | `anio_mes_compra`, `price`, `order_id` | Serie mensual: unidades, órdenes, ingresos e ingreso promedio por ítem |

**Ajustes de alcance respecto a la AE2 (justificados):**

1. **Ventas efectivas.** Las unidades y los ingresos se calculan solo con ítems de órdenes que no fueron canceladas
   ni quedaron no disponibles (`canceled`, `unavailable`): esas órdenes no representan una venta. Las órdenes en curso
   (`shipped`, `invoiced`, `processing`, `approved`) sí se cuentan como venta, porque el producto fue comprado y no se anuló.
2. **Tiempo de entrega.** Solo se calcula para ítems de órdenes entregadas (`delivered`) que tienen fecha de entrega.
3. **Ingresos** = suma de `price` (precio de los productos, sin flete), igual que en la AE2. La moneda se asume en reales
   brasileños (BRL) porque es un marketplace de Brasil; la ficha original no la declara explícitamente.
4. **Limitación que se mantiene:** no hay costos ni márgenes, así que no se puede hablar de rentabilidad, y el análisis
   describe lo ocurrido sin explicar sus causas.

## 2. Preparar el entorno y cargar el dataset de la AE2

Se usan las mismas clases del proyecto: `HFDatasetLoader` (obtención) y `DatasetSummarizer` (resumen `.txt`).

In [1]:
import warnings

warnings.filterwarnings("ignore")

import sys
from pathlib import Path

import pandas as pd
from datasets.utils.logging import disable_progress_bar, set_verbosity_error
from huggingface_hub.utils import logging as hf_logging
from IPython.display import display

set_verbosity_error()
hf_logging.set_verbosity_error()
disable_progress_bar()

ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / "src" / "clases_po").exists():
    ROOT = ROOT.parent

sys.path.insert(0, str(ROOT / "src"))

from clases_po import DatasetSummarizer, HFDatasetLoader

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 100)

Se cargan las mismas seis tablas y la misma revisión de la AE2, y se construye el mismo DataFrame de ítems (left join
desde los ítems). `df_original` se conserva **sin modificar** para comparar los cambios al final de la transformación.

In [2]:
DATASET_ID = "bulutttt/olist-raw-data"
REVISION = "5acf1f37d7197d4979e714721a1ec13a7c3da728"
SPLIT = "train"
ARCHIVOS = {
    "items": "olist_order_items_dataset.csv",
    "ordenes": "olist_orders_dataset.csv",
    "productos": "olist_products_dataset.csv",
    "categorias": "product_category_name_translation.csv",
    "clientes": "olist_customers_dataset.csv",
    "vendedores": "olist_sellers_dataset.csv",
}

tablas = HFDatasetLoader.load_tables(DATASET_ID, ARCHIVOS, split=SPLIT, revision=REVISION)

df_original = (
    tablas["items"]
    .merge(tablas["ordenes"], on="order_id", how="left")
    .merge(tablas["productos"], on="product_id", how="left")
    .merge(tablas["categorias"], on="product_category_name", how="left")
    .merge(tablas["clientes"], on="customer_id", how="left")
    .merge(tablas["vendedores"], on="seller_id", how="left")
)
assert len(df_original) == len(tablas["items"]), "Las uniones duplicaron o perdieron filas"

df = df_original.copy()  # sobre esta copia se aplican las transformaciones
print(f"DataFrame de la AE2: {df_original.shape[0]} filas y {df_original.shape[1]} columnas")

DataFrame de la AE2: 112650 filas y 30 columnas


In [3]:
def perfil(data: pd.DataFrame) -> pd.DataFrame:
    """Tipo, nulos y valores únicos por columna."""
    return pd.DataFrame(
        {
            "tipo": data.dtypes.astype(str),
            "nulos": data.isna().sum(),
            "% nulos": (data.isna().mean() * 100).round(2),
            "valores_unicos": data.nunique(),
        }
    )


perfil_antes = perfil(df_original)
perfil_antes

,tipo,nulos,% nulos,valores_unicos
order_id,object,0,0.00,98666
order_item_id,int64,0,0.00,21
product_id,object,0,0.00,32951
seller_id,object,0,0.00,3095
shipping_limit_date,object,0,0.00,93318
price,float64,0,0.00,5968
freight_value,float64,0,0.00,6999
customer_id,object,0,0.00,98666
order_status,object,0,0.00,7
order_purchase_timestamp,object,0,0.00,98112


## 3. Transformación y validación de los datos

Cada paso indica qué cambia, por qué y cuántos registros afecta. Los controles que no detectan problemas también se
informan. Los cambios se registran en `bitacora` para resumirlos al final.

In [4]:
bitacora = []


def registrar(paso, decision, registros):
    bitacora.append({"paso": paso, "decision": decision, "registros_afectados": int(registros)})

### 3.1 Duplicados

Se revisan filas idénticas y la llave del ítem (`order_id`, `order_item_id`), que debe ser única.

In [5]:
dup_filas = int(df.duplicated().sum())
dup_llave = int(df.duplicated(["order_id", "order_item_id"]).sum())
print(f"Filas completamente duplicadas: {dup_filas}")
print(f"Llaves (order_id, order_item_id) repetidas: {dup_llave}")
registrar("Duplicados", "Control sin hallazgos: no hay filas ni llaves de ítem duplicadas; no se elimina nada", dup_filas + dup_llave)

Filas completamente duplicadas: 0
Llaves (order_id, order_item_id) repetidas: 0


### 3.2 Tipos de datos: fechas

Las fechas llegan como texto (`object`). Se convierten a `datetime` con `errors="coerce"` y se verifica que la
conversión no genere nulos nuevos (un nulo nuevo indicaría un texto con formato inválido).

In [6]:
COLUMNAS_FECHA = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
    "shipping_limit_date",
]

conversion = []
for columna in COLUMNAS_FECHA:
    nulos_antes = int(df[columna].isna().sum())
    df[columna] = pd.to_datetime(df[columna], errors="coerce")
    nulos_despues = int(df[columna].isna().sum())
    conversion.append(
        {
            "columna": columna,
            "tipo_nuevo": str(df[columna].dtype),
            "nulos_antes": nulos_antes,
            "nulos_despues": nulos_despues,
            "fechas_invalidas": nulos_despues - nulos_antes,
        }
    )

conversion = pd.DataFrame(conversion)
assert conversion["fechas_invalidas"].sum() == 0
registrar("Tipos de fecha", "6 columnas de texto convertidas a datetime64; 0 fechas inválidas", len(df))
conversion

,columna,tipo_nuevo,nulos_antes,nulos_despues,fechas_invalidas
0,order_purchase_timestamp,datetime64[ns],0,0,0
1,order_approved_at,datetime64[ns],15,15,0
2,order_delivered_carrier_date,datetime64[ns],1194,1194,0
3,order_delivered_customer_date,datetime64[ns],2454,2454,0
4,order_estimated_delivery_date,datetime64[ns],0,0,0
5,shipping_limit_date,datetime64[ns],0,0,0


### 3.3 Consistencia de fechas y del estado de la orden

Se revisan reglas de calidad que las fechas deben cumplir para calcular el tiempo de entrega.

In [7]:
entregado = df["order_status"] == "delivered"
tiene_fecha_entrega = df["order_delivered_customer_date"].notna()

controles_fechas = pd.DataFrame(
    {
        "control": [
            "Entrega al cliente anterior a la compra",
            "Fecha estimada anterior a la compra",
            "Orden entregada (delivered) sin fecha de entrega",
            "Orden no entregada con fecha de entrega",
        ],
        "registros": [
            int((df["order_delivered_customer_date"] < df["order_purchase_timestamp"]).sum()),
            int((df["order_estimated_delivery_date"] < df["order_purchase_timestamp"]).sum()),
            int((entregado & ~tiene_fecha_entrega).sum()),
            int((~entregado & tiene_fecha_entrega).sum()),
        ],
    }
)
display(controles_fechas)
print("Estados de las órdenes no entregadas que tienen fecha de entrega:")
print(df.loc[~entregado & tiene_fecha_entrega, "order_status"].value_counts().to_dict())

,control,registros
0,Entrega al cliente anterior a la compra,0
1,Fecha estimada anterior a la compra,0
2,Orden entregada (delivered) sin fecha de entrega,8
3,Orden no entregada con fecha de entrega,7


Estados de las órdenes no entregadas que tienen fecha de entrega:
{'canceled': 7}


**Tratamiento:**

- No hay entregas ni fechas estimadas anteriores a la compra.
- Los ítems entregados sin fecha de entrega **no se eliminan** (son ventas válidas); solo quedan sin tiempo de entrega.
- Los ítems con fecha de entrega cuya orden figura como cancelada se excluyen junto con todas las órdenes canceladas
  en el paso 3.6, porque no son ventas efectivas.

In [8]:
registrar(
    "Entregadas sin fecha",
    "Se conservan como venta; quedan sin dias_entrega ni entrega_tardia",
    controles_fechas.loc[2, "registros"],
)

### 3.4 Valores numéricos

`price` y `freight_value` ya son numéricos (`float64`). Se revisan valores imposibles (precio menor o igual a cero o
flete negativo) y valores extremos.

In [9]:
controles_numericos = pd.DataFrame(
    {
        "control": [
            "price <= 0",
            "freight_value < 0",
            "freight_value = 0 (envío sin costo)",
            "price por encima del percentil 99",
        ],
        "registros": [
            int((df["price"] <= 0).sum()),
            int((df["freight_value"] < 0).sum()),
            int((df["freight_value"] == 0).sum()),
            int((df["price"] > df["price"].quantile(0.99)).sum()),
        ],
    }
)
display(controles_numericos)
print(f"Precio: mínimo {df['price'].min():.2f}, mediana {df['price'].median():.2f}, máximo {df['price'].max():.2f}")
registrar("Valores numéricos", "Control sin hallazgos: no hay precios <= 0 ni fletes negativos; los valores altos se conservan", 0)

,control,registros
0,price <= 0,0
1,freight_value < 0,0
2,freight_value = 0 (envío sin costo),383
3,price por encima del percentil 99,1117


Precio: mínimo 0.85, mediana 74.99, máximo 6735.00


No hay precios ni fletes imposibles. El flete en cero es posible (envío gratis) y no afecta los ingresos, que se miden
sin flete. Los precios altos corresponden a productos costosos reales y **se conservan**: eliminarlos subestimaría los
ingresos. Para el tiempo de entrega se reporta la mediana, que es poco sensible a valores extremos.

### 3.5 Categorías: faltantes y normalización

Se crea `categoria_producto` a partir de `product_category_name_english`, en tres pasos:

1. **Categorías sin traducción:** dos categorías en portugués no tienen traducción en la tabla de traducciones. Se
   traducen manualmente para no perder esos ítems.
2. **Errores de escritura:** algunos nombres en inglés tienen errores (`fashio_`, `costruction_`, `confort`). Se
   corrigen para que las tablas sean legibles; no se fusionan categorías distintas.
3. **Sin categoría:** los productos sin categoría en el origen se agrupan como `sin_categoria`, como se definió en la
   AE2. No se eliminan porque son ventas reales.

In [10]:
TRADUCCIONES_FALTANTES = {
    "pc_gamer": "pc_gamer",
    "portateis_cozinha_e_preparadores_de_alimentos": "portable_kitchen_food_preparers",
}
CORRECCIONES_ESCRITURA = {
    "fashio_female_clothing": "fashion_female_clothing",
    "costruction_tools_garden": "construction_tools_garden",
    "costruction_tools_tools": "construction_tools_tools",
    "home_confort": "home_comfort",
}

sin_traduccion = df["product_category_name"].notna() & df["product_category_name_english"].isna()
sin_categoria = df["product_category_name"].isna()
con_error = df["product_category_name_english"].isin(CORRECCIONES_ESCRITURA)

df["categoria_producto"] = (
    df["product_category_name_english"]
    .fillna(df["product_category_name"].map(TRADUCCIONES_FALTANTES))
    .replace(CORRECCIONES_ESCRITURA)
    .fillna("sin_categoria")
)

registrar("Categorías sin traducción", "Traducción manual de 2 categorías (pc_gamer, portateis_cozinha...)", sin_traduccion.sum())
registrar("Categorías mal escritas", "Corrección de 4 nombres con errores de escritura", con_error.sum())
registrar("Sin categoría", "Nulos agrupados como 'sin_categoria'", sin_categoria.sum())

print(f"Ítems sin traducción: {sin_traduccion.sum()} | con nombre mal escrito: {con_error.sum()} | sin categoría: {sin_categoria.sum()}")
print(f"Categorías antes: {df['product_category_name_english'].nunique()} (+ nulos) | después: {df['categoria_producto'].nunique()}")
assert df["categoria_producto"].isna().sum() == 0

Ítems sin traducción: 24 | con nombre mal escrito: 823 | sin categoría: 1603
Categorías antes: 71 (+ nulos) | después: 74


Se revisa también que los textos categóricos usados en el análisis no tengan espacios sobrantes ni mayúsculas
inconsistentes.

In [11]:
inconsistencias_texto = {
    "customer_state": int((df["customer_state"] != df["customer_state"].str.strip().str.upper()).sum()),
    "order_status": int((df["order_status"] != df["order_status"].str.strip().str.lower()).sum()),
    "categoria_producto": int((df["categoria_producto"] != df["categoria_producto"].str.strip().str.lower()).sum()),
}
print(inconsistencias_texto)
print(f"Estados del cliente: {df['customer_state'].nunique()} -> {sorted(df['customer_state'].unique())}")
registrar("Formato de textos", "Control sin hallazgos: estados, estado de la orden y categoría sin espacios ni mayúsculas inconsistentes", sum(inconsistencias_texto.values()))

{'customer_state': 0, 'order_status': 0, 'categoria_producto': 0}
Estados del cliente: 27 -> ['AC', 'AL', 'AM', 'AP', 'BA', 'CE', 'DF', 'ES', 'GO', 'MA', 'MG', 'MS', 'MT', 'PA', 'PB', 'PE', 'PI', 'PR', 'RJ', 'RN', 'RO', 'RR', 'RS', 'SC', 'SE', 'SP', 'TO']


### 3.6 Selección de registros: ventas efectivas

Se excluyen los ítems de órdenes **canceladas** (`canceled`) y **no disponibles** (`unavailable`), porque no representan
una venta realizada. Los demás estados se conservan.

In [12]:
ESTADOS_EXCLUIDOS = ["canceled", "unavailable"]

print("Ítems por estado de la orden:")
print(df["order_status"].value_counts().to_string())

excluir = df["order_status"].isin(ESTADOS_EXCLUIDOS)
filas_antes = len(df)
df = df.loc[~excluir].copy()
registrar("Ventas efectivas", "Se excluyen ítems de órdenes canceled y unavailable", excluir.sum())
print(f"\nÍtems excluidos: {excluir.sum()} | filas: {filas_antes} -> {len(df)}")

Ítems por estado de la orden:
order_status
delivered      110197
shipped          1185
canceled          542
invoiced          359
processing        357
unavailable         7
approved            3

Ítems excluidos: 549 | filas: 112650 -> 112101


### 3.7 Selección de columnas

Se conservan solo las columnas que usan las preguntas de negocio. Se descartan: identificadores que no se usan en las
tablas (`customer_id`, `customer_unique_id`, `seller_id`, `product_id`), fechas intermedias del proceso logístico
(`order_approved_at`, `order_delivered_carrier_date`, `shipping_limit_date`), atributos físicos del producto,
ubicación del vendedor y ciudad/código postal del cliente (el análisis regional es por estado) y las dos columnas de
categoría originales, reemplazadas por `categoria_producto`.

In [13]:
COLUMNAS_ANALISIS = [
    "order_id",
    "order_item_id",
    "order_status",
    "order_purchase_timestamp",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
    "price",
    "freight_value",
    "categoria_producto",
    "customer_state",
]
columnas_descartadas = [c for c in df.columns if c not in COLUMNAS_ANALISIS]
df = df[COLUMNAS_ANALISIS].copy()
registrar("Selección de columnas", f"Se conservan {len(COLUMNAS_ANALISIS)} columnas y se descartan {len(columnas_descartadas)}", len(columnas_descartadas))
print(f"Columnas descartadas ({len(columnas_descartadas)}): {columnas_descartadas}")

Columnas descartadas (21): ['product_id', 'seller_id', 'shipping_limit_date', 'customer_id', 'order_approved_at', 'order_delivered_carrier_date', 'product_category_name', 'product_name_lenght', 'product_description_lenght', 'product_photos_qty', 'product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm', 'product_category_name_english', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'seller_zip_code_prefix', 'seller_city', 'seller_state']


### 3.8 Tipos categóricos

Las columnas con pocos valores distintos que se usan para agrupar se convierten a tipo `category`.

In [14]:
for columna in ["order_status", "categoria_producto", "customer_state"]:
    df[columna] = df[columna].astype("category")
df.dtypes

order_id                                 object
order_item_id                             int64
order_status                           category
order_purchase_timestamp         datetime64[ns]
order_delivered_customer_date    datetime64[ns]
order_estimated_delivery_date    datetime64[ns]
price                                   float64
freight_value                           float64
categoria_producto                     category
customer_state                         category
dtype: object

## 4. Enriquecimiento

Se incorporan variables calculadas a partir de las columnas existentes y una clasificación regional externa.

| Variable | Fórmula o regla | Significado | Pregunta |
|---|---|---|---|
| `anio_mes_compra` | Año y mes de `order_purchase_timestamp` (formato `AAAA-MM`) | Periodo de la compra | P5 |
| `dias_entrega` | (`order_delivered_customer_date` − `order_purchase_timestamp`) en días, con 2 decimales; solo órdenes `delivered` con fecha de entrega | Días entre la compra y la recepción | P3, P4 |
| `entrega_tardia` | `True` si el día de entrega es posterior a `order_estimated_delivery_date`; vacío si no aplica | Incumplimiento de la fecha estimada | P3, P4 |
| `region_cliente` | Región de Brasil del estado del cliente (división oficial del IBGE) | Agrupa los 27 estados en 5 regiones | P2, P3 |

In [15]:
df["anio_mes_compra"] = df["order_purchase_timestamp"].dt.to_period("M").astype(str)

aplica_entrega = (df["order_status"] == "delivered") & df["order_delivered_customer_date"].notna()

df["dias_entrega"] = (
    (df["order_delivered_customer_date"] - df["order_purchase_timestamp"]).dt.total_seconds() / 86400
).round(2).where(aplica_entrega)

df["entrega_tardia"] = (
    df["order_delivered_customer_date"].dt.normalize() > df["order_estimated_delivery_date"]
).astype("boolean").where(aplica_entrega, pd.NA)

print(f"Ítems con tiempo de entrega calculado: {aplica_entrega.sum()} de {len(df)}")
print(f"Ítems sin tiempo de entrega (no entregados o sin fecha): {(~aplica_entrega).sum()}")

Ítems con tiempo de entrega calculado: 110189 de 112101
Ítems sin tiempo de entrega (no entregados o sin fecha): 1912


**Región del cliente.** Se combina una tabla de referencia con la región de cada estado, según la división regional
oficial del Instituto Brasileño de Geografía y Estadística (IBGE, [Grandes Regiões](https://www.ibge.gov.br/apps/quadrogeografico/pdf/qg_2025_010_GrandReg.pdf)). La llave de relación es la sigla del estado
(`customer_state` = `estado`). Se verifica que la tabla tenga una fila por estado (relación muchos a uno) y que la unión
no multiplique filas ni importes.

In [16]:
REGIONES_IBGE = {
    "Norte": ["AC", "AM", "AP", "PA", "RO", "RR", "TO"],
    "Nordeste": ["AL", "BA", "CE", "MA", "PB", "PE", "PI", "RN", "SE"],
    "Centro-Oeste": ["DF", "GO", "MS", "MT"],
    "Sudeste": ["ES", "MG", "RJ", "SP"],
    "Sur": ["PR", "RS", "SC"],
}
regiones = pd.DataFrame(
    [(estado, region) for region, estados in REGIONES_IBGE.items() for estado in estados],
    columns=["estado", "region_cliente"],
)
assert regiones["estado"].is_unique and len(regiones) == 27

filas_antes, ingresos_antes = len(df), df["price"].sum()
df["customer_state"] = df["customer_state"].astype(str)
df = df.merge(regiones, left_on="customer_state", right_on="estado", how="left", validate="many_to_one").drop(columns="estado")
df["customer_state"] = df["customer_state"].astype("category")
df["region_cliente"] = df["region_cliente"].astype("category")

assert len(df) == filas_antes, "La unión multiplicó filas"
assert round(df["price"].sum(), 2) == round(ingresos_antes, 2), "La unión alteró los ingresos"
assert df["region_cliente"].isna().sum() == 0, "Hay estados sin región"
registrar("Región del cliente", "Unión con tabla IBGE por sigla del estado; sin filas nuevas ni estados sin región", 0)
print(f"Filas: {filas_antes} -> {len(df)} | ingresos sin cambios: {df['price'].sum():,.2f}")
df["region_cliente"].value_counts()

Filas: 112101 -> 112101 | ingresos sin cambios: 13,494,400.74


region_cliente
Sudeste         76981
Sur             16091
Nordeste        10377
Centro-Oeste     6592
Norte            2060
Name: count, dtype: int64

### 4.1 Comparación antes y después

Se compara el DataFrame de la AE2 (`df_original`) con el DataFrame final transformado y enriquecido.

In [17]:
comparacion = pd.DataFrame(
    {
        "AE2 (original)": [df_original.shape[0], df_original.shape[1], int(df_original.isna().sum().sum()),
                           int((df_original.dtypes == "object").sum()), 0],
        "AE3 (final)": [df.shape[0], df.shape[1], int(df.isna().sum().sum()),
                        int((df.dtypes == "object").sum()),
                        int(df.dtypes.astype(str).str.startswith("datetime").sum())],
    },
    index=["filas", "columnas", "valores nulos", "columnas de texto (object)", "columnas de fecha (datetime)"],
)
comparacion

,AE2 (original),AE3 (final)
filas,112650,112101
columnas,30,14
valores nulos,11774,5736
columnas de texto (object),18,2
columnas de fecha (datetime),0,3


In [18]:
perfil_despues = perfil(df)
comparacion_columnas = perfil_antes.join(perfil_despues, how="outer", lsuffix="_antes", rsuffix="_despues")
comparacion_columnas = comparacion_columnas.loc[
    [c for c in df_original.columns if c in comparacion_columnas.index]
    + [c for c in df.columns if c not in df_original.columns]
]
comparacion_columnas[["tipo_antes", "tipo_despues", "nulos_antes", "nulos_despues"]]

,tipo_antes,tipo_despues,nulos_antes,nulos_despues
order_id,object,object,0.0,0.0
order_item_id,int64,int64,0.0,0.0
product_id,object,NaN,0.0,NaN
seller_id,object,NaN,0.0,NaN
shipping_limit_date,object,NaN,0.0,NaN
price,float64,float64,0.0,0.0
freight_value,float64,float64,0.0,0.0
customer_id,object,NaN,0.0,NaN
order_status,object,category,0.0,0.0
order_purchase_timestamp,object,datetime64[ns],0.0,0.0


Los nulos que quedan en el DataFrame final son esperados: `order_delivered_customer_date`, `dias_entrega` y
`entrega_tardia` están vacíos para los ítems que no se han entregado o no tienen fecha de entrega.

In [19]:
bitacora_df = pd.DataFrame(bitacora)
bitacora_df

,paso,decision,registros_afectados
0,Duplicados,Control sin hallazgos: no hay filas ni llaves ...,0
1,Tipos de fecha,6 columnas de texto convertidas a datetime64; ...,112650
2,Entregadas sin fecha,Se conservan como venta; quedan sin dias_entre...,8
3,Valores numéricos,Control sin hallazgos: no hay precios <= 0 ni ...,0
4,Categorías sin traducción,"Traducción manual de 2 categorías (pc_gamer, p...",24
5,Categorías mal escritas,Corrección de 4 nombres con errores de escritura,823
6,Sin categoría,Nulos agrupados como 'sin_categoria',1603
7,Formato de textos,"Control sin hallazgos: estados, estado de la o...",0
8,Ventas efectivas,Se excluyen ítems de órdenes canceled y unavai...,549
9,Selección de columnas,Se conservan 10 columnas y se descartan 21,21


## 5. Respuestas a las preguntas de negocio

Unidades = cantidad de ítems vendidos (filas). Ingresos = suma de `price` en BRL. Participación = proporción del total
en porcentaje. Las tablas de tiempo de entrega usan solo los ítems entregados con fecha.

In [20]:
TOTAL_UNIDADES = len(df)
TOTAL_INGRESOS = df["price"].sum()
TOTAL_ORDENES = df["order_id"].nunique()
entregas = df[df["dias_entrega"].notna()]

print(f"Unidades vendidas: {TOTAL_UNIDADES:,}")
print(f"Órdenes: {TOTAL_ORDENES:,}")
print(f"Ingresos (BRL): {TOTAL_INGRESOS:,.2f}")
print(f"Ítems con tiempo de entrega: {len(entregas):,}")
print(f"Mediana general de días de entrega: {entregas['dias_entrega'].median():.2f}")
print(f"Entregas tardías: {entregas['entrega_tardia'].mean() * 100:.2f} %")


def tabla_ventas(data, grupo):
    """Unidades, órdenes, ingresos, participación e ingreso promedio por grupo."""
    t = (
        data.groupby(grupo, observed=True)
        .agg(unidades=("order_id", "size"), ordenes=("order_id", "nunique"), ingresos_brl=("price", "sum"))
        .sort_values("ingresos_brl", ascending=False)
    )
    t["participacion_unidades_pct"] = t["unidades"] / t["unidades"].sum() * 100
    t["participacion_ingresos_pct"] = t["ingresos_brl"] / t["ingresos_brl"].sum() * 100
    t["participacion_ingresos_acum_pct"] = t["participacion_ingresos_pct"].cumsum()
    t["ingreso_promedio_item_brl"] = t["ingresos_brl"] / t["unidades"]
    t.insert(0, "ranking_ingresos", range(1, len(t) + 1))
    return t.round(2).reset_index()


def tabla_entregas(data, grupo):
    """Ítems entregados, mediana y promedio de días y % de entregas tardías por grupo."""
    t = (
        data.groupby(grupo, observed=True)
        .agg(
            items_entregados=("dias_entrega", "size"),
            mediana_dias_entrega=("dias_entrega", "median"),
            promedio_dias_entrega=("dias_entrega", "mean"),
            entregas_tardias=("entrega_tardia", "sum"),
        )
        .sort_values("mediana_dias_entrega", ascending=False)
    )
    t["entregas_tardias"] = t["entregas_tardias"].astype(int)
    t["pct_entregas_tardias"] = t["entregas_tardias"] / t["items_entregados"] * 100
    return t.round(2).reset_index()

Unidades vendidas: 112,101


Órdenes: 98,199
Ingresos (BRL): 13,494,400.74
Ítems con tiempo de entrega: 110,189
Mediana general de días de entrega: 10.18
Entregas tardías: 6.59 %


### P1. ¿Qué categorías de producto concentran las unidades vendidas y los ingresos?

Nivel de detalle: categoría de producto. Ordenada por ingresos. Unidades en ítems; ingresos en BRL; participaciones en %.

In [21]:
tabla_p1 = tabla_ventas(df, "categoria_producto")
print(f"Categorías: {len(tabla_p1)}")
tabla_p1.head(10)

Categorías: 74


,categoria_producto,ranking_ingresos,unidades,ordenes,ingresos_brl,participacion_unidades_pct,participacion_ingresos_pct,participacion_ingresos_acum_pct,ingreso_promedio_item_brl
0,health_beauty,1,9634,8800,1255695.13,8.59,9.31,9.31,130.34
1,watches_gifts,2,5970,5604,1198185.21,5.33,8.88,18.18,200.70
2,bed_bath_table,3,11097,9399,1035964.06,9.90,7.68,25.86,93.36
3,sports_leisure,4,8590,7673,979740.92,7.66,7.26,33.12,114.06
4,computers_accessories,5,7781,6654,904322.02,6.94,6.70,39.82,116.22
5,furniture_decor,6,8298,6425,727465.05,7.40,5.39,45.21,87.67
6,housewares,7,6915,5847,626825.80,6.17,4.65,49.86,90.65
7,cool_stuff,8,3779,3616,620770.49,3.37,4.60,54.46,164.27
8,auto,9,4204,3872,586585.73,3.75,4.35,58.81,139.53
9,garden_tools,10,4328,3505,481009.94,3.86,3.56,62.37,111.14


In [22]:
top10 = tabla_p1.head(10)
print(f"Las 10 categorías con más ingresos suman {top10['participacion_ingresos_pct'].sum():.2f} % de los ingresos "
      f"y {top10['participacion_unidades_pct'].sum():.2f} % de las unidades.")
print(f"Categorías necesarias para llegar al 80 % de los ingresos: {(tabla_p1['participacion_ingresos_acum_pct'] < 80).sum() + 1}")
lider_unidades = tabla_p1.sort_values("unidades", ascending=False).iloc[0]
print(f"Categoría con más unidades: {lider_unidades['categoria_producto']} ({lider_unidades['unidades']:,} ítems)")
print("Ingreso promedio por ítem en el top 10:")
print(top10.set_index("categoria_producto")["ingreso_promedio_item_brl"].sort_values(ascending=False).to_string())

Las 10 categorías con más ingresos suman 62.38 % de los ingresos y 62.97 % de las unidades.
Categorías necesarias para llegar al 80 % de los ingresos: 18
Categoría con más unidades: bed_bath_table (11,097 ítems)
Ingreso promedio por ítem en el top 10:
categoria_producto
watches_gifts            200.70
cool_stuff               164.27
auto                     139.53
health_beauty            130.34
computers_accessories    116.22
sports_leisure           114.06
garden_tools             111.14
bed_bath_table            93.36
housewares                90.65
furniture_decor           87.67


**Interpretación P1.** Las ventas están repartidas entre muchas categorías, pero se concentran en un grupo reducido:
las 10 categorías con más ingresos suman el 62,38 % de los ingresos y el 62,97 % de las unidades, y 18 de las 74
categorías alcanzan el 80 % de los ingresos. `health_beauty` lidera en ingresos (1.255.695,13 BRL; 9,31 %), mientras
que `bed_bath_table` es la categoría con más unidades (11.097 ítems) pero ocupa el tercer lugar en ingresos porque su
ingreso promedio por ítem es bajo (93,36 BRL). `watches_gifts` muestra el caso contrario: con menos unidades (5.970)
es la segunda en ingresos, por su ingreso promedio de 200,70 BRL por ítem.

**Utilidad para la necesidad:** permite priorizar categorías según el objetivo comercial: volumen (unidades) o valor
(ingresos), que no siempre coinciden.

**Limitaciones:** los ingresos son el precio de venta sin flete ni descuentos; sin costos no se puede afirmar qué
categoría es más rentable. `sin_categoria` agrupa 1.603 ítems cuya categoría no se conoce.

### P2. ¿Cómo se distribuyen las unidades, los ingresos y el ingreso promedio por ítem entre los estados del cliente?

Nivel de detalle: estado del cliente (27 unidades federativas), con su región. Ordenada por ingresos.

In [23]:
tabla_p2 = tabla_ventas(df, ["customer_state", "region_cliente"])
tabla_p2

,customer_state,region_cliente,ranking_ingresos,unidades,ordenes,ingresos_brl,participacion_unidades_pct,participacion_ingresos_pct,participacion_ingresos_acum_pct,ingreso_promedio_item_brl
0,SP,Sudeste,1,47153,41125,5163867.22,42.06,38.27,38.27,109.51
1,RJ,Sudeste,2,14509,12697,1811623.42,12.94,13.43,51.69,124.86
2,MG,Sudeste,3,13071,11496,1573508.20,11.66,11.66,63.35,120.38
3,RS,Sur,4,6211,5415,742559.78,5.54,5.50,68.85,119.56
4,PR,Sur,5,5719,4982,676883.06,5.10,5.02,73.87,118.36
5,SC,Sur,6,4161,3599,518578.28,3.71,3.84,77.71,124.63
6,BA,Nordeste,7,3785,3344,507108.83,3.38,3.76,81.47,133.98
7,DF,Centro-Oeste,8,2397,2120,300886.45,2.14,2.23,83.70,125.53
8,GO,Centro-Oeste,9,2323,1998,287870.46,2.07,2.13,85.83,123.92
9,ES,Sudeste,10,2248,2018,273532.13,2.01,2.03,87.86,121.68


In [24]:
por_region = tabla_ventas(df, "region_cliente")[
    ["region_cliente", "unidades", "ingresos_brl", "participacion_unidades_pct", "participacion_ingresos_pct", "ingreso_promedio_item_brl"]
]
print("Subtotal por región (calculado con la misma función):")
display(por_region)
print("Estados con mayor ingreso promedio por ítem:")
print(tabla_p2.nlargest(5, "ingreso_promedio_item_brl")[["customer_state", "region_cliente", "unidades", "ingreso_promedio_item_brl"]].to_string(index=False))

Subtotal por región (calculado con la misma función):


,region_cliente,unidades,ingresos_brl,participacion_unidades_pct,participacion_ingresos_pct,ingreso_promedio_item_brl
0,Sudeste,76981,8822530.97,68.67,65.38,114.61
1,Sur,16091,1938021.12,14.35,14.36,120.44
2,Nordeste,10377,1538209.28,9.26,11.40,148.23
3,Centro-Oeste,6592,861825.09,5.88,6.39,130.74
4,Norte,2060,333814.28,1.84,2.47,162.05


Estados con mayor ingreso promedio por ítem:
customer_state region_cliente  unidades  ingreso_promedio_item_brl
            PB       Nordeste       600                     191.46
            AL       Nordeste       444                     180.89
            AC          Norte        92                     173.73
            RO          Norte       277                     166.18
            PA          Norte      1079                     165.73


**Interpretación P2.** Las ventas se concentran en el Sudeste: São Paulo (SP) aporta el 42,06 % de las unidades y el
38,27 % de los ingresos, y junto con Río de Janeiro (RJ) y Minas Gerais (MG) suman el 63,35 % de los ingresos. Por
región, el Sudeste reúne el 65,38 % de los ingresos y el Norte solo el 2,47 %. En cambio, el ingreso promedio por ítem
es mayor en los estados con menos ventas: Paraíba (PB, 191,46 BRL) y Alagoas (AL, 180,89 BRL) superan ampliamente a SP
(109,51 BRL), y las regiones Norte (162,05 BRL) y Nordeste (148,23 BRL) tienen los promedios más altos.

**Utilidad para la necesidad:** SP, RJ y MG son los mercados de mayor volumen; los estados del Norte y Nordeste
compran menos, pero cada ítem vale más en promedio.

**Limitaciones:** la diferencia en el ingreso promedio describe lo observado, no su causa (por ejemplo, mezcla de
categorías distinta). Los estados con pocas unidades (RR, 51 ítems; AP, 82) tienen promedios menos estables.

### P3. ¿Cómo varía el tiempo de entrega y el porcentaje de entregas tardías entre los estados del cliente?

Nivel de detalle: estado del cliente. Tiempo en días; entregas tardías en %. Ordenada por mediana de días (de mayor a menor).

In [25]:
tabla_p3 = tabla_entregas(entregas, ["customer_state", "region_cliente"])
tabla_p3

,customer_state,region_cliente,items_entregados,mediana_dias_entrega,promedio_dias_entrega,entregas_tardias,pct_entregas_tardias
0,AM,Norte,163,26.00,26.39,5,3.07
1,RR,Norte,46,24.92,28.23,5,10.87
2,AP,Norte,81,24.09,28.21,3,3.70
3,AL,Nordeste,427,22.07,24.49,89,20.84
4,PA,Norte,1054,21.04,23.75,119,11.29
5,MA,Nordeste,800,19.31,21.65,144,18.00
6,PB,Nordeste,586,18.36,20.59,63,10.75
7,AC,Norte,91,18.33,20.72,3,3.30
8,CE,Nordeste,1426,18.09,20.99,194,13.60
9,RO,Norte,273,18.05,19.74,11,4.03


In [26]:
print("Por región (referencia):")
display(tabla_entregas(entregas, "region_cliente"))

Por región (referencia):


,region_cliente,items_entregados,mediana_dias_entrega,promedio_dias_entrega,entregas_tardias,pct_entregas_tardias
0,Norte,2018,20.10,22.60,176,8.72
1,Nordeste,10087,17.28,19.91,1266,12.55
2,Centro-Oeste,6480,13.22,14.94,419,6.47
3,Sur,15879,12.07,13.99,912,5.74
4,Sudeste,75725,8.68,10.68,4491,5.93


**Interpretación P3.** El tiempo de entrega crece con la distancia al Sudeste, donde se ubica la mayoría de los
vendedores (AE2: el 71,3 % de los ítems se despacha desde SP). SP tiene la menor mediana (7,21 días), mientras que
Amazonas (AM, 26,00), Roraima (RR, 24,92) y Amapá (AP, 24,09) superan los 24 días. Por región, la mediana va de 8,68 días
en el Sudeste a 20,10 en el Norte. Sin embargo, el **porcentaje de entregas tardías** no sigue el mismo orden: el Norte
tiene entregas largas pero pocas tardías (8,72 %), porque la fecha estimada ya es más amplia; el Nordeste tiene el mayor
porcentaje de tardías (12,55 %), con Alagoas (20,84 %) y Maranhão (18,00 %) a la cabeza. Río de Janeiro llama la
atención: con una mediana de 11,99 días, el 11,62 % de sus entregas llegó después de la fecha estimada.

**Utilidad para la necesidad:** distingue dos problemas logísticos distintos: estados con entregas largas (Norte) y
estados donde no se cumple la fecha prometida (Nordeste y RJ).

**Limitaciones:** solo incluye ítems entregados con fecha (110.189); se mide en ítems, por lo que una orden con varios
ítems pesa más; no explica las causas de las demoras. Los estados con pocos ítems (RR, 46) tienen porcentajes inestables.

### P4. ¿Cómo varía el tiempo de entrega y el porcentaje de entregas tardías entre las categorías de producto?

Nivel de detalle: categoría de producto. La tabla exportada incluye todas las categorías; para interpretar se muestran
las 10 categorías con más ingresos (P1), porque las categorías con muy pocos ítems entregados producen medidas poco estables.

In [27]:
tabla_p4 = tabla_entregas(entregas, "categoria_producto")
print(f"Categorías: {len(tabla_p4)} | con menos de 100 ítems entregados: {(tabla_p4['items_entregados'] < 100).sum()}")
tabla_p4_top10 = (
    tabla_p4.set_index("categoria_producto")
    .loc[top10["categoria_producto"]]
    .sort_values("mediana_dias_entrega", ascending=False)
    .reset_index()
)
tabla_p4_top10

Categorías: 74 | con menos de 100 ítems entregados: 20


,categoria_producto,items_entregados,mediana_dias_entrega,promedio_dias_entrega,entregas_tardias,pct_entregas_tardias
0,garden_tools,4268,11.84,13.71,280,6.56
1,computers_accessories,7643,11.14,13.23,496,6.49
2,furniture_decor,8160,10.76,12.88,574,7.03
3,bed_bath_table,10953,10.39,12.81,770,7.03
4,cool_stuff,3718,10.38,12.36,217,5.84
5,watches_gifts,5857,10.28,12.66,422,7.21
6,sports_leisure,8430,10.09,12.15,532,6.31
7,auto,4139,9.78,12.23,291,7.03
8,health_beauty,9465,9.50,11.98,716,7.56
9,housewares,6795,8.87,10.92,340,5.00


In [28]:
mayores_100 = tabla_p4[tabla_p4["items_entregados"] >= 100]
print("Categorías con 100 o más ítems entregados y mayor % de entregas tardías:")
print(mayores_100.nlargest(5, "pct_entregas_tardias").to_string(index=False))
print("\nRango de la mediana de días entre categorías con 100 o más ítems: "
      f"{mayores_100['mediana_dias_entrega'].min():.2f} a {mayores_100['mediana_dias_entrega'].max():.2f}")

Categorías con 100 o más ítems entregados y mayor % de entregas tardías:
     categoria_producto  items_entregados  mediana_dias_entrega  promedio_dias_entrega  entregas_tardias  pct_entregas_tardias
                  audio               362                 10.66                  13.32                42                 11.60
     christmas_supplies               150                 13.04                  15.74                15                 10.00
fashion_underwear_beach               127                 10.69                  13.71                12                  9.45
           home_comfort               429                 11.14                  13.53                40                  9.32
        books_technical               263                  8.98                  10.65                21                  7.98

Rango de la mediana de días entre categorías con 100 o más ítems: 7.01 a 19.02


**Interpretación P4.** Entre las categorías el tiempo de entrega varía menos que entre estados. En las 10 categorías con
más ingresos, la mediana va de 8,87 días (`housewares`) a 11,84 días (`garden_tools`) y el porcentaje de entregas tardías
se mantiene entre 5,00 % y 7,56 %, cerca del promedio general (6,59 %). Considerando las categorías con al menos 100 ítems
entregados, la mediana va de 7,01 días (`food`) a 19,02 días (`office_furniture`, productos voluminosos) y el mayor porcentaje de
tardías es el de `audio` (11,60 %).

**Utilidad para la necesidad:** la categoría influye menos que la región en el desempeño de la entrega; las acciones
logísticas tendrían más alcance si se orientan por región que por categoría.

**Limitaciones:** 20 categorías tienen menos de 100 ítems entregados y sus medidas son poco estables. La categoría y la
región se mezclan (una categoría puede venderse más en estados lejanos), y este análisis descriptivo no separa ese efecto.

### P5. ¿Cómo evolucionan mes a mes las unidades vendidas y los ingresos?

Nivel de detalle: mes de compra (`AAAA-MM`). Se incluyen todos los meses del periodo; un mes sin ventas aparece con cero
para que la serie quede continua al graficarla en la AE4.

In [29]:
meses = pd.period_range(df["order_purchase_timestamp"].min(), df["order_purchase_timestamp"].max(), freq="M").astype(str)
tabla_p5 = (
    df.groupby("anio_mes_compra")
    .agg(unidades=("order_id", "size"), ordenes=("order_id", "nunique"), ingresos_brl=("price", "sum"))
    .reindex(meses, fill_value=0)
    .rename_axis("anio_mes_compra")
)
tabla_p5["ingreso_promedio_item_brl"] = (tabla_p5["ingresos_brl"] / tabla_p5["unidades"]).fillna(0)
tabla_p5["participacion_ingresos_pct"] = tabla_p5["ingresos_brl"] / tabla_p5["ingresos_brl"].sum() * 100
tabla_p5 = tabla_p5.round(2).reset_index()
tabla_p5

,anio_mes_compra,unidades,ordenes,ingresos_brl,ingreso_promedio_item_brl,participacion_ingresos_pct
0,2016-09,5,2,207.86,41.57,0.00
1,2016-10,342,290,44507.30,130.14,0.33
2,2016-11,0,0,0.00,0.00,0.00
3,2016-12,1,1,10.90,10.90,0.00
4,2017-01,953,787,120098.27,126.02,0.89
5,2017-02,1936,1718,244959.35,126.53,1.82
6,2017-03,2975,2617,368341.32,123.81,2.73
7,2017-04,2660,2377,353842.98,133.02,2.62
8,2017-05,4106,3640,503159.19,122.54,3.73
9,2017-06,3571,3205,429916.61,120.39,3.19


In [30]:
completos = tabla_p5[tabla_p5["unidades"] >= 100]
mes_max = completos.loc[completos["ingresos_brl"].idxmax()]
print(f"Meses con menos de 100 ítems: {tabla_p5.loc[tabla_p5['unidades'] < 100, 'anio_mes_compra'].tolist()}")
print(f"Mes con más ingresos: {mes_max['anio_mes_compra']} ({mes_max['ingresos_brl']:,.2f} BRL; {mes_max['unidades']:,} ítems)")
por_anio = df.assign(anio=df["order_purchase_timestamp"].dt.year)
ene_ago = por_anio[por_anio["order_purchase_timestamp"].dt.month <= 8].groupby("anio")["price"].agg(["size", "sum"])
print("Enero a agosto de cada año (unidades, ingresos):")
print(ene_ago.round(2).to_string())
crec = (ene_ago.loc[2018, "sum"] / ene_ago.loc[2017, "sum"] - 1) * 100
print(f"Variación de ingresos ene-ago 2018 vs. ene-ago 2017: {crec:.2f} %")

Meses con menos de 100 ítems: ['2016-09', '2016-11', '2016-12', '2018-09']
Mes con más ingresos: 2017-11 (1,003,862.14 BRL; 8,626 ítems)
Enero a agosto de cada año (unidades, ingresos):
       size         sum
anio                   
2017  25576  3080850.81
2018  61135  7341037.41
Variación de ingresos ene-ago 2018 vs. ene-ago 2017: 138.28 %


**Interpretación P5.** Las ventas crecieron durante el periodo: entre enero y agosto de 2018 los ingresos fueron un
138,28 % mayores que en los mismos meses de 2017 (7.341.037,41 frente a 3.080.850,81 BRL). El mes con más ingresos fue
noviembre de 2017 (1.003.862,14 BRL; 8.626 ítems), y desde enero de 2018 las ventas se estabilizan entre unos 7.000 y
8.200 ítems por mes. El ingreso promedio por ítem se mantiene estable (entre 109,57 y 133,02 BRL en los meses con datos
suficientes), por lo que el crecimiento se explica por más unidades vendidas y no por precios más altos.

**Utilidad para la necesidad:** muestra la tendencia del negocio y el pico de noviembre, útil para planificar campañas
y capacidad logística.

**Limitaciones:** los meses 2016-09, 2016-11, 2016-12 y 2018-09 tienen menos de 100 ítems (2016-11 no tiene ventas) y
no son comparables: reflejan el inicio y el corte del dataset, no caídas reales de las ventas. Los ingresos no se
ajustan por inflación.

### 5.1 Verificación de totales y porcentajes

Las tablas por grupo deben sumar el total del DataFrame final y las participaciones deben sumar 100 %.

In [31]:
TOL = 0.5  # tolerancia por redondeo a 2 decimales

for nombre, t in {"P1": tabla_p1, "P2": tabla_p2, "P5": tabla_p5}.items():
    assert t["unidades"].sum() == TOTAL_UNIDADES, nombre
    assert abs(t["ingresos_brl"].sum() - TOTAL_INGRESOS) < TOL, nombre
    assert abs(t["participacion_ingresos_pct"].sum() - 100) < 0.1, nombre
for nombre, t in {"P3": tabla_p3, "P4": tabla_p4}.items():
    assert t["items_entregados"].sum() == len(entregas), nombre
    assert t["entregas_tardias"].sum() == int(entregas["entrega_tardia"].sum()), nombre
assert tabla_p5["ordenes"].sum() == TOTAL_ORDENES

print("Verificación correcta: todas las tablas suman los totales del DataFrame final.")
pd.DataFrame(
    {
        "unidades": [t["unidades"].sum() for t in (tabla_p1, tabla_p2, tabla_p5)],
        "ingresos_brl": [round(t["ingresos_brl"].sum(), 2) for t in (tabla_p1, tabla_p2, tabla_p5)],
        "suma_participacion_ingresos_pct": [round(t["participacion_ingresos_pct"].sum(), 2) for t in (tabla_p1, tabla_p2, tabla_p5)],
    },
    index=["P1 categorías", "P2 estados", "P5 meses"],
)

Verificación correcta: todas las tablas suman los totales del DataFrame final.


,unidades,ingresos_brl,suma_participacion_ingresos_pct
P1 categorías,112101,13494400.74,100.01
P2 estados,112101,13494400.74,100.04
P5 meses,112101,13494400.74,100.01


## 6. Exportar el dataset preparado y las tablas de resultados

Las rutas son relativas al notebook (`AE3/notebooks/`). Las carpetas se crean si no existen y los archivos se
sobrescriben en cada ejecución. Después de exportar, cada archivo se lee de nuevo y se compara con el DataFrame del
notebook.

In [32]:
DIR_DATOS = Path("..") / "datos"
DIR_RESULTADOS = Path("..") / "resultados"
DIR_DATOS.mkdir(parents=True, exist_ok=True)
DIR_RESULTADOS.mkdir(parents=True, exist_ok=True)

RUTA_DATASET = DIR_DATOS / "olist_items_preparado.csv"
TABLAS = {
    "tabla_p1_categorias.csv": tabla_p1,
    "tabla_p2_estados.csv": tabla_p2,
    "tabla_p3_entrega_estados.csv": tabla_p3,
    "tabla_p4_entrega_categorias.csv": tabla_p4,
    "tabla_p5_mensual.csv": tabla_p5,
}

df.to_csv(RUTA_DATASET, index=False, encoding="utf-8")
for archivo, tabla in TABLAS.items():
    tabla.to_csv(DIR_RESULTADOS / archivo, index=False, encoding="utf-8")

exportados = [RUTA_DATASET] + [DIR_RESULTADOS / a for a in TABLAS]
pd.DataFrame(
    {
        "archivo": [str(r.resolve().relative_to(ROOT)).replace("\\", "/") for r in exportados],
        "filas": [len(df)] + [len(t) for t in TABLAS.values()],
        "tamano_kb": [round(r.stat().st_size / 1024, 1) for r in exportados],
    }
)

,archivo,filas,tamano_kb
0,AE3/datos/olist_items_preparado.csv,112101,16637.8
1,AE3/resultados/tabla_p1_categorias.csv,74,4.5
2,AE3/resultados/tabla_p2_estados.csv,27,1.7
3,AE3/resultados/tabla_p3_entrega_estados.csv,27,1.1
4,AE3/resultados/tabla_p4_entrega_categorias.csv,74,3.1
5,AE3/resultados/tabla_p5_mensual.csv,25,1.0


In [33]:
# Verificar que los archivos exportados coinciden con el notebook
for archivo, tabla in TABLAS.items():
    leida = pd.read_csv(DIR_RESULTADOS / archivo)
    pd.testing.assert_frame_equal(leida, tabla.astype({c: str for c in tabla.select_dtypes("category").columns}),
                                  check_dtype=False, check_exact=False, atol=0.01)

dataset_leido = pd.read_csv(RUTA_DATASET)
assert dataset_leido.shape == df.shape
assert list(dataset_leido.columns) == list(df.columns)
assert abs(dataset_leido["price"].sum() - TOTAL_INGRESOS) < TOL
print("Verificación correcta: los archivos .csv coinciden con los resultados del notebook.")

Verificación correcta: los archivos .csv coinciden con los resultados del notebook.


## 7. Generar automáticamente el resumen `.txt` del DataFrame final

`DatasetSummarizer` recibe el **DataFrame final transformado y enriquecido** (`df`) y escribe
`AE3/resultados/resumen_dataframe_final.txt`: origen de los datos, filas y columnas, nombres y tipos de columnas,
valores nulos, duplicados, estadísticas descriptivas, valores más frecuentes y rango de fechas. Se regenera en cada ejecución.

In [34]:
RUTA_RESUMEN = DIR_RESULTADOS / "resumen_dataframe_final.txt"

metadatos = {
    "Dataset": DATASET_ID,
    "Enlace": f"https://huggingface.co/datasets/{DATASET_ID}",
    "Revisión": REVISION,
    "Split": SPLIT,
    "Archivos": ", ".join(ARCHIVOS.values()),
    "Tabla externa": "Regiones de Brasil por estado (IBGE), unida por la sigla del estado",
    "Unidad de análisis": "Ítem de orden (un producto vendido dentro de una orden)",
    "Registros": f"Ítems de órdenes no canceladas ni no disponibles ({len(df)} de {len(df_original)})",
    "Dataset preparado": "AE3/datos/olist_items_preparado.csv",
}

resumen = DatasetSummarizer(df, titulo="Resumen del DataFrame final - Olist (AE3)", metadatos=metadatos)
ruta = resumen.guardar(RUTA_RESUMEN)
print(f"Resumen guardado en: {ruta.resolve().relative_to(ROOT)}")

Resumen guardado en: AE3\resultados\resumen_dataframe_final.txt


In [35]:
contenido = ruta.read_text(encoding="utf-8")

assert f"Filas: {df.shape[0]}" in contenido
assert f"Columnas: {df.shape[1]}" in contenido
assert all(columna in contenido for columna in df.columns)
assert f"Total de valores nulos: {int(df.isna().sum().sum())}" in contenido
print("Verificación correcta: el resumen corresponde al DataFrame final de esta ejecución.\n")

print(contenido)

Verificación correcta: el resumen corresponde al DataFrame final de esta ejecución.

RESUMEN DEL DATAFRAME FINAL - OLIST (AE3)
Generado: 2026-11-08 20:26:10

Origen de los datos
-------------------
- Dataset: bulutttt/olist-raw-data
- Enlace: https://huggingface.co/datasets/bulutttt/olist-raw-data
- Revisión: 5acf1f37d7197d4979e714721a1ec13a7c3da728
- Split: train
- Archivos: olist_order_items_dataset.csv, olist_orders_dataset.csv, olist_products_dataset.csv, product_category_name_translation.csv, olist_customers_dataset.csv, olist_sellers_dataset.csv
- Tabla externa: Regiones de Brasil por estado (IBGE), unida por la sigla del estado
- Unidad de análisis: Ítem de orden (un producto vendido dentro de una orden)
- Registros: Ítems de órdenes no canceladas ni no disponibles (112101 de 112650)
- Dataset preparado: AE3/datos/olist_items_preparado.csv

Dimensiones
-----------
Filas: 112101
Columnas: 14
Filas duplicadas: 0
Memoria aproximada: 21,795.4 KB

Columnas y tipos
----------------
- 

## 8. Conclusiones del avance

- La transformación dejó un DataFrame de **112.101 ítems y 14 columnas**, con fechas en tipo `datetime`, categorías
  normalizadas, solo ventas efectivas y sin duplicados. Las variables incorporadas (`anio_mes_compra`, `dias_entrega`,
  `entrega_tardia`, `region_cliente`) permitieron responder las cinco preguntas de negocio.
- **Ventas:** se concentran en pocas categorías (10 categorías generan el 62,38 % de los ingresos) y en el Sudeste
  (65,38 % de los ingresos), en especial SP. Volumen y valor no coinciden: hay categorías y estados con pocas unidades
  pero alto ingreso promedio por ítem.
- **Entregas:** el tiempo de entrega depende sobre todo de la región (mediana de 8,68 días en el Sudeste y 20,10 en el
  Norte); el incumplimiento de la fecha estimada es mayor en el Nordeste (12,55 %) y en RJ (11,62 %).
- **Queda sin resolver:** las causas de las diferencias, la rentabilidad (no hay costos) y la verificación de la moneda.
- Las tablas exportadas en `AE3/resultados/` quedan listas para su representación gráfica en la AE4.